# Experiments

In [7]:
# importing libraries
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:
SPLITS = Path("../data/splits")

## Popularity Baseline

In the training set, what are the most rated books? Doesn't matter if they are liked or not, but what are the most rated.

And to check - show this "model" some books. If those books exist in what it "recommends", don't recommend that. And then, check metrics.

In [3]:
train = pd.read_parquet(SPLITS / "train_split.parquet", columns=["user_id", "work_id"])

In [10]:
train.shape

(102089653, 2)

In [4]:
pop = (train.work_id.value_counts().rename("n_interactions").reset_index().rename(columns={"index": "work_id"}))

In [5]:
print(f"{len(pop):,} works ranked")
pop.head(20)

991,014 works ranked


,work_id,n_interactions
0,4640799,311275
1,2792775,286922
2,3212258,231664
3,3275794,209731
4,2402163,193763
5,6231171,193053
6,3046572,185148
7,245494,182440
8,6171458,182049
9,2809203,178722


In [6]:
books = pd.read_parquet("../data/interim/english_works_all.parquet",columns=["work_id", "title", "ratings_total"])

pop.head(20).merge(books, on="work_id", how="left")[["title", "n_interactions", "ratings_total"]]

,title,n_interactions,ratings_total
0,Harry Potter and the Sorcerer's Stone (Harry P...,311275,4970387
1,"The Hunger Games (The Hunger Games, #1)",286922,5064668
2,"Twilight (Twilight, #1)",231664,3991256
3,To Kill a Mockingbird,209731,3399207
4,Harry Potter and the Prisoner of Azkaban (Harr...,193763,2016007
5,Harry Potter and the Chamber of Secrets (Harry...,193053,1950555
6,Harry Potter and the Goblet of Fire (Harry Pot...,185148,1909895
7,The Great Gatsby,182440,2841340
8,"Catching Fire (The Hunger Games, #2)",182049,2013187
9,Harry Potter and the Order of the Phoenix (Har...,178722,1873246


### Note

Yeah, they all look pretty popular.

In [8]:
val = pd.read_parquet(SPLITS / "val_split.parquet",
                      columns=["user_id", "work_id", "rating"])

In [9]:
val.shape

(1805359, 3)

In [11]:
rng = np.random.default_rng(0)
pos = val[val.rating >= 4]
keep = rng.random(len(pos)) < 0.5

fold_in = pd.concat([val[val.rating < 4], pos[keep]])
target = pos[~keep]

seen = fold_in.groupby("user_id").work_id.apply(set)
truth = target.groupby("user_id").work_id.apply(set)

print(f"{len(truth):,} users, median {truth.apply(len).median():.0f} targets")

10,000 users, median 33 targets


In [12]:
K = 10
ranking = pop.work_id.to_numpy()
head = ranking[:K + seen.apply(len).max()]

recs = {u: [b for b in head if b not in seen.get(u, set())][:K]
        for u in truth.index}

In [13]:
def evaluate(recs, truth, k):
    disc = 1 / np.log2(np.arange(2, k + 2))
    rec, ndcg = [], []
    for u, items in recs.items():
        t = truth[u]
        hits = np.array([b in t for b in items[:k]])
        n = min(len(t), k)
        rec.append(hits.sum() / n)
        ndcg.append((hits * disc).sum() / disc[:n].sum())
    return np.mean(rec), np.mean(ndcg)

r, n = evaluate(recs, truth, K)
print(f"recall@{K} {r:.4f}   ndcg@{K} {n:.4f}")

recall@10 0.1977   ndcg@10 0.2214


In [14]:
flat = [b for v in recs.values() for b in v]
print(f"{len(set(flat)):,} distinct books served to {len(recs):,} users")

38 distinct books served to 10,000 users
